# 03. Circuit Resource Analysis and Transform Validation

This notebook separates two claims: (1) the Qiskit QFT convention matches the NumPy FFT split step used for the harmonic oscillator, and (2) resource counts are small-system exact-synthesis or analytical estimates, not asymptotic optimal decompositions. The infinite-well resource model is reported for a quantum-sine-transform implementation rather than a periodic QFT circuit.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qftsplit.core import (
    build_infinite_well_circuit,
    build_periodic_resource_circuit,
    configure_matplotlib,
    dirichlet_midpoint_grid,
    draw_and_save_circuit,
    fft_momentum_grid,
    fidelity_summary,
    harmonic_potential,
    periodic_grid,
    plot_convergence,
    plot_density_snapshots,
    plot_fidelity_vs_gate_count,
    plot_fidelity,
    qft_split_circuit_validation,
    run_harmonic_case,
    run_infinite_well_case,
    save_dataframe,
    save_publication_figure,
    sine_mode_energies,
    sine_transform_validation,
    transpile_and_extract_metrics,
)

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

In [2]:
N = 64
hbar = 1.0
m = 1.0
omega = 1.0

harmonic_domain = (-8.0, 8.0)
harmonic_t_max = 2.0 * np.pi
harmonic_r_default = 200

well_L = 10.0
well_t_max = 6.0
well_r_default = 700

basis_gates = ["rx", "ry", "rz", "cx"]

In [3]:
validation_records = []

x_h, dx_h = periodic_grid(harmonic_domain[0], harmonic_domain[1], N)
p_h = fft_momentum_grid(N, dx=dx_h, hbar=hbar)
dt_h = harmonic_t_max / harmonic_r_default
position_phase_h = np.exp(-0.5j * harmonic_potential(x_h, mass=m, omega=omega) * dt_h / hbar)
momentum_phase_h = np.exp(-1j * (p_h**2 / (2.0 * m)) * dt_h / hbar)
validation_records.append(qft_split_circuit_validation(position_phase_h, momentum_phase_h))

dt_w = well_t_max / well_r_default
validation_records.append(sine_transform_validation(N, length=well_L, mass=m, hbar=hbar, dt=dt_w))

validation_df = pd.DataFrame(validation_records)
save_dataframe(validation_df, TABLES_DIR, "circuit_validation.csv")
display(Markdown("## Transform Validation"))
display(validation_df)

if not validation_df["passed"].all():
    raise RuntimeError("At least one transform validation failed.")

## Transform Validation

,validation,n_qubits,direct_l2_error,phase_adjusted_l2_error,passed,grid_size
0,qiskit_qft_matches_numpy_fft_step,6.0,8.429068e-15,7.258797e-15,True,NaN
1,dst_matrix_matches_scipy_sine_step,NaN,3.934573e-16,NaN,True,64.0


In [4]:
harmonic_logical = build_periodic_resource_circuit(position_phase_h, momentum_phase_h)
harmonic_transpiled, harmonic_metrics, harmonic_breakdown = transpile_and_extract_metrics(harmonic_logical, basis_gates)

n_qubits = int(np.log2(N))
qst_extension_qubits = n_qubits + 2
qft_controlled_phases = qst_extension_qubits * (qst_extension_qubits - 1) // 2
kinetic_controlled_phases = n_qubits * (n_qubits - 1) // 2
qst_blocks_per_step = 2
total_controlled_phases = qst_blocks_per_step * qft_controlled_phases + kinetic_controlled_phases
well_metrics = {
    "single_step_1q_count": int(qst_blocks_per_step * qst_extension_qubits + 3 * total_controlled_phases + n_qubits),
    "single_step_2q_count": int(2 * total_controlled_phases),
    "single_step_depth": int(qst_blocks_per_step * qst_extension_qubits**2 + 4 * kinetic_controlled_phases),
}

single_step_metrics_df = pd.DataFrame(
    [
        {
            "system": "harmonic_oscillator",
            "N": N,
            "n_qubits": n_qubits,
            "dt": dt_h,
            **harmonic_metrics,
            "transform": "periodic_QFT",
            "synthesis_model": "Qiskit_exact_generic_DiagonalGate_small_N_upper_bound",
        },
        {
            "system": "infinite_well",
            "N": N,
            "n_qubits": n_qubits,
            "dt": dt_w,
            **well_metrics,
            "transform": "Dirichlet_quantum_sine_transform",
            "synthesis_model": "analytical_all_to_all_QST_via_QFT_extension_estimate",
        },
    ]
)

gate_breakdown_records = [
    {"system": "harmonic_oscillator", "gate_name": gate_name, "count": gate_count}
    for gate_name, gate_count in sorted(harmonic_breakdown.items())
]
gate_breakdown_records.extend(
    [
        {"system": "infinite_well", "gate_name": "QST_blocks", "count": qst_blocks_per_step},
        {"system": "infinite_well", "gate_name": "QFT_extension_controlled_phases_per_QST", "count": qft_controlled_phases},
        {"system": "infinite_well", "gate_name": "kinetic_quadratic_phase_controlled_phases", "count": kinetic_controlled_phases},
        {"system": "infinite_well", "gate_name": "estimated_cx", "count": well_metrics["single_step_2q_count"]},
    ]
)
gate_breakdown_df = pd.DataFrame(gate_breakdown_records)

save_dataframe(single_step_metrics_df, TABLES_DIR, "circuit_single_step_metrics.csv")
save_dataframe(gate_breakdown_df, TABLES_DIR, "circuit_gate_breakdown.csv")

display(Markdown("## Single-Step Metrics"))
display(single_step_metrics_df)
display(Markdown("## Gate Breakdown"))
display(gate_breakdown_df)

C:\Users\Tilock\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


## Single-Step Metrics

,system,N,n_qubits,dt,single_step_1q_count,single_step_2q_count,single_step_depth,transform,synthesis_model
0,harmonic_oscillator,64,6,0.031416,171,264,312,periodic_QFT,Qiskit_exact_generic_DiagonalGate_small_N_uppe...
1,infinite_well,64,6,0.008571,235,142,188,Dirichlet_quantum_sine_transform,analytical_all_to_all_QST_via_QFT_extension_es...


## Gate Breakdown

,system,gate_name,count
0,harmonic_oscillator,barrier,4
1,harmonic_oscillator,cx,264
2,harmonic_oscillator,rx,6
3,harmonic_oscillator,ry,12
4,harmonic_oscillator,rz,153
5,infinite_well,QST_blocks,2
6,infinite_well,QFT_extension_controlled_phases_per_QST,28
7,infinite_well,kinetic_quadratic_phase_controlled_phases,15
8,infinite_well,estimated_cx,142


In [5]:
draw_and_save_circuit(harmonic_logical, FIGURES_DIR, "harmonic_single_step_logical_circuit", scale=0.75, fold=40)
draw_and_save_circuit(harmonic_transpiled, FIGURES_DIR, "harmonic_single_step_transpiled_circuit", scale=0.55, fold=50)

infinite_well_logical = build_infinite_well_circuit(grid_size=N, length=well_L, mass=m, hbar=hbar, dt=dt_w)
draw_and_save_circuit(infinite_well_logical, FIGURES_DIR, "infinite_well_single_step_logical_circuit_qiskit", scale=0.75, fold=40)

infinite_well_transpiled, _, _ = transpile_and_extract_metrics(infinite_well_logical, basis_gates)
draw_and_save_circuit(infinite_well_transpiled, FIGURES_DIR, "infinite_well_single_step_transpiled_circuit_qiskit", scale=0.55, fold=50)



def save_block_diagram(stem: str, title: str, blocks: list[str]) -> None:
    fig, axis = plt.subplots(figsize=(10.0, 2.0), constrained_layout=True)
    axis.set_axis_off()

    box_style = {
        "boxstyle": "round,pad=0.6",
        "facecolor": "#ebf5fb", 
        "edgecolor": "#2874a6", 
        "linewidth": 2.0,
    }

    if len(blocks) == 3:
        x_positions = np.linspace(0.25, 0.75, len(blocks))
    else:
        x_positions = np.linspace(0.12, 0.88, len(blocks))

    for x_pos, label in zip(x_positions, blocks):
        axis.text(
            x_pos, 0.5, label,
            ha="center", va="center",
            fontsize=13, fontweight="bold", color="#1b4f72",
            bbox=box_style,
        )

    for i in range(len(blocks) - 1):
        left_text = blocks[i]
        right_text = blocks[i+1]

        left_hw = max(0.04, len(left_text) * 0.005 + 0.03)
        right_hw = max(0.04, len(right_text) * 0.005 + 0.03)

        axis.annotate(
            "", 
            xy=(x_positions[i+1] - right_hw, 0.5), 
            xytext=(x_positions[i] + left_hw, 0.5), 
            arrowprops={
                "arrowstyle": "-|>,head_length=0.8,head_width=0.4", 
                "lw": 2.5, 
                "color": "#2874a6"
            }
        )

    axis.set_title(title, fontsize=15, fontweight="bold", pad=15)
    save_publication_figure(fig, FIGURES_DIR, stem)
    plt.close(fig)

save_block_diagram(
    "infinite_well_single_step_logical_circuit",
    "Infinite-well Dirichlet spectral step",
    ["QST", "T(dt)", "QST^-1"],
)
save_block_diagram(
    "infinite_well_single_step_transpiled_circuit",
    "Analytical QST resource model",
    ["QFT extension", "odd/sine symmetry", "quadratic phase", "inverse QST"],
)

print("Saved circuit and resource-model diagrams.")

Saved circuit and resource-model diagrams.


In [6]:
fidelity_tables = {
    "harmonic_oscillator": TABLES_DIR / "harmonic_fidelity_vs_r.csv",
    "infinite_well": TABLES_DIR / "infinite_well_fidelity_vs_r.csv",
}

resource_totals_frames = []
for system_key, fidelity_path in fidelity_tables.items():
    if not fidelity_path.exists():
        raise FileNotFoundError(f"Missing {fidelity_path.name}. Run the corresponding dynamics notebook first.")
    fidelity_df = pd.read_csv(fidelity_path)
    metrics_row = single_step_metrics_df.loc[single_step_metrics_df["system"] == system_key].iloc[0]
    totals_df = fidelity_df.copy()
    totals_df["single_step_1q_count"] = int(metrics_row["single_step_1q_count"])
    totals_df["single_step_2q_count"] = int(metrics_row["single_step_2q_count"])
    totals_df["single_step_depth"] = int(metrics_row["single_step_depth"])
    totals_df["synthesis_model"] = metrics_row["synthesis_model"]
    totals_df["total_1q_count"] = totals_df["single_step_1q_count"] * totals_df["r"]
    totals_df["total_2q_count"] = totals_df["single_step_2q_count"] * totals_df["r"]
    totals_df["total_gate_count"] = totals_df["total_1q_count"] + totals_df["total_2q_count"]
    totals_df["estimated_total_depth"] = totals_df["single_step_depth"] * totals_df["r"]
    resource_totals_frames.append(totals_df)
    save_dataframe(totals_df, TABLES_DIR, f"{system_key}_resource_totals_vs_r.csv")

resource_totals_df = pd.concat(resource_totals_frames, ignore_index=True)
save_dataframe(resource_totals_df, TABLES_DIR, "resource_totals_vs_r.csv")
display(Markdown("## Resource Totals Across Fidelity Sweeps"))
display(resource_totals_df)

## Resource Totals Across Fidelity Sweeps

,system,r,dt,final_time_fidelity,minimum_fidelity,mean_fidelity,single_step_1q_count,single_step_2q_count,single_step_depth,synthesis_model,total_1q_count,total_2q_count,total_gate_count,estimated_total_depth
0,harmonic_oscillator,50,0.125664,0.999927,0.999905,0.999968,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,8550,13200,21750,15600
1,harmonic_oscillator,100,0.062832,0.999995,0.999994,0.999998,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,17100,26400,43500,31200
2,harmonic_oscillator,150,0.041888,0.999999,0.999999,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,25650,39600,65250,46800
3,harmonic_oscillator,200,0.031416,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,34200,52800,87000,62400
4,harmonic_oscillator,250,0.025133,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,42750,66000,108750,78000
5,harmonic_oscillator,300,0.020944,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,51300,79200,130500,93600
6,harmonic_oscillator,400,0.015708,1.000000,1.000000,1.000000,171,264,312,Qiskit_exact_generic_DiagonalGate_small_N_uppe...,68400,105600,174000,124800
7,infinite_well,100,0.060000,1.000000,1.000000,1.000000,235,142,188,analytical_all_to_all_QST_via_QFT_extension_es...,23500,14200,37700,18800
8,infinite_well,200,0.030000,1.000000,1.000000,1.000000,235,142,188,analytical_all_to_all_QST_via_QFT_extension_es...,47000,28400,75400,37600
9,infinite_well,300,0.020000,1.000000,1.000000,1.000000,235,142,188,analytical_all_to_all_QST_via_QFT_extension_es...,70500,42600,113100,56400


In [7]:
harmonic_resource_df = resource_totals_df.loc[resource_totals_df["system"] == "harmonic_oscillator"].reset_index(drop=True)
well_resource_df = resource_totals_df.loc[resource_totals_df["system"] == "infinite_well"].reset_index(drop=True)

figures = [
    (plot_convergence(harmonic_resource_df, "r", "total_2q_count", "Harmonic oscillator CX count", "Trotter steps, r"), "harmonic_gate_counts_vs_steps"),
    (plot_convergence(well_resource_df, "r", "total_2q_count", "Infinite-well estimated CX count", "Trotter steps, r"), "infinite_well_gate_counts_vs_steps"),
    (plot_fidelity_vs_gate_count(harmonic_resource_df, "total_2q_count", "Total CX count", "Harmonic fidelity vs CX count"), "harmonic_fidelity_vs_two_qubit_gate_count"),
    (plot_fidelity_vs_gate_count(harmonic_resource_df, "total_gate_count", "Total 1q + 2q gate count", "Harmonic fidelity vs total gate count"), "harmonic_fidelity_vs_total_gate_count"),
    (plot_fidelity_vs_gate_count(well_resource_df, "total_2q_count", "Estimated total CX count", "Infinite-well fidelity vs CX count"), "infinite_well_fidelity_vs_two_qubit_gate_count"),
    (plot_fidelity_vs_gate_count(well_resource_df, "total_gate_count", "Estimated total 1q + 2q gate count", "Infinite-well fidelity vs total gate count"), "infinite_well_fidelity_vs_total_gate_count"),
]
for figure, stem in figures:
    save_publication_figure(figure, FIGURES_DIR, stem)
    plt.close(figure)

print("Saved gate-count and fidelity-vs-gate-count figures.")

Saved gate-count and fidelity-vs-gate-count figures.
